# Framework Showdown: LangGraph vs MAF -- Architecture, Trade-offs, and When to Use Which

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Articulate the core philosophy** behind each framework -- graph-based vs imperative async.
2. **Compare real code** from this repository side by side: tool definitions, agent construction, state management.
3. **Evaluate trade-offs** in persistence, human-in-the-loop, async support, and MCP integration.
4. **Choose the right framework** for a given project using a concrete decision framework.

---

## How This Notebook Fits In

This workshop implements **11+ agents in both LangGraph and MAF** (Microsoft Agent Framework). Every agent in `single-agents/langgraph/` has a counterpart in `single-agents/maf/`, and the full multi-agent router exists in both `agent-patterns/Langgraph/` and `agent-patterns/MAF/`.

This notebook is a capstone: it synthesizes everything you have seen across all previous notebooks and scripts into a systematic comparison. The code cells below are **illustrative excerpts** drawn from the actual repository files, not standalone runnable agents.

## Section 1: Two Philosophies of Agent Construction

The two frameworks represent fundamentally different answers to the same question: *how should you express the control flow of an AI agent?*

**LangGraph -- "Define the graph, let the framework run it"**

LangGraph models every agent as a directed graph. Nodes are functions. Edges define the transitions between them. Conditional edges let you branch based on state. You compile the graph once, then invoke it. The framework handles the execution loop, checkpointing, and resumption.

This is a *declarative* approach: you describe **what** the topology looks like, and LangGraph figures out **how** to run it.

**MAF -- "Write async functions, compose agents"**

MAF takes an *imperative* approach. You create an agent object, hand it tools and instructions, and call `await agent.run()`. The ReAct loop is handled internally. Control flow is standard Python: `if/else`, `for` loops, `asyncio.gather`. There is no graph to define.

This is closer to how most Python developers already write code -- top-down, step-by-step, with async/await for concurrency.

| Dimension | LangGraph | MAF |
|-----------|-----------|-----|
| Mental model | Directed graph of nodes and edges | Async Python functions |
| Control flow | Declared via `add_edge` / `add_conditional_edges` | Standard Python (`if`, `for`, `await`) |
| Execution | Framework-managed graph traversal | Developer-managed async calls |
| Learning curve | Graph concepts + LangChain ecosystem | Familiar Python async patterns |

## Section 2: The Simplest Agent -- Side by Side

Both frameworks implement the same basic ReAct agent with the same two tools (`get_weather` and `simple_search`). Compare the structural differences.

**Source files:**
- LangGraph: `single-agents/langgraph/basic_react_agent.py`
- MAF: `single-agents/maf/basic_react_agent.py`

In [1]:
# ===========================================================================
# LangGraph: Basic ReAct Agent (condensed from basic_react_agent.py)
# ===========================================================================
# Key imports -- note the graph-specific vocabulary
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage

# 1. Define typed state -- every node reads and writes this
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Define tools with the @tool decorator
@tool
def get_weather(location: str = "San Francisco"):
    """Call to get the current weather."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

@tool
def simple_search(query: str):
    """Search for information about AI, Python, or LangGraph."""
    query = query.lower()
    if "ai" in query:
        return "AI stands for Artificial Intelligence."
    elif "python" in query:
        return "Python is used for AI, web, and automation."
    return "No results found."

# 3. Bind tools to the LLM
tools = [get_weather, simple_search]
tool_node = ToolNode(tools)
# llm_with_tools = llm.bind_tools(tools)

# 4. Define the agent node (reasoning step)
def agent(state: State):
    system_message = SystemMessage(content="You are a helpful assistant.")
    messages = [system_message] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# 5. Define the routing function
def tool_router(state: State):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 6. Build the graph -- this is the core of LangGraph
builder = StateGraph(State)
builder.add_node("agent", agent)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tool_router, ["tools", END])
builder.add_edge("tools", "agent")
graph = builder.compile(checkpointer=MemorySaver())

print("LangGraph agent: ~40 lines of structural code")
print("Components: State TypedDict, @tool decorators, ToolNode, StateGraph, conditional edges")

LangGraph agent: ~40 lines of structural code
Components: State TypedDict, @tool decorators, ToolNode, StateGraph, conditional edges


In [2]:
# ===========================================================================
# MAF: Basic ReAct Agent (condensed from basic_react_agent.py)
# ===========================================================================
import asyncio
from agent_framework.openai import OpenAIChatClient

# 1. Define tools as plain Python functions -- no decorator needed
def simple_search(query: str) -> str:
    query = query.lower()
    if "ai" in query:
        return "AI stands for Artificial Intelligence."
    elif "python" in query:
        return "Python is used for AI, web, and automation."
    return "No results found."

def get_weather(location: str = "San Francisco") -> str:
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

# 2. Create the agent -- one call does everything
async def main():
    agent = OpenAIChatClient().as_agent(
        name="My Azure Agent",
        description="An agent with tools.",
        instructions="You are a helpful assistant with access to tools.",
        tools=[get_weather, simple_search],
    )
    response = await agent.run("What's the weather in SF?")
    print("Assistant:", response.text)

print("MAF agent: ~15 lines of structural code")
print("Components: plain functions, OpenAIChatClient().as_agent(), await agent.run()")

MAF agent: ~15 lines of structural code
Components: plain functions, OpenAIChatClient().as_agent(), await agent.run()


### Analysis

The LangGraph version requires roughly 3x more structural code than MAF. But that code is not wasted -- it buys you:

- **Explicit state**: `State(TypedDict)` with the `add_messages` reducer. You know exactly what data flows between nodes.
- **Visible topology**: `add_node`, `add_edge`, `add_conditional_edges` spell out the entire execution graph. You can visualize it, test individual nodes, and reason about it statically.
- **Built-in checkpointing**: `MemorySaver()` gives you persistence and replay with one line.

MAF trades all of that for brevity. The ReAct loop, tool dispatch, and message management are handled internally. This is ideal when the standard loop is all you need.

## Section 3: Tool Definition Patterns

Tool definitions are where the frameworks are most similar -- and where the differences are most instructive.

In [3]:
# ===========================================================================
# LangGraph tool definition
# ===========================================================================
from langchain_core.tools import tool

@tool
def get_weather_lg(city: str) -> str:
    """Get the current weather for a city.
    The LLM reads this docstring to decide when to call the tool.
    The @tool decorator converts this into a LangChain Tool object
    with a name, description, and JSON schema derived from the signature.
    """
    return f"Sunny in {city}"

# The decorator creates a structured tool object:
print(f"Name:        {get_weather_lg.name}")
print(f"Description: {get_weather_lg.description}")
print(f"Schema:      {get_weather_lg.args_schema.schema()}")

Name:        get_weather_lg
Description: Get the current weather for a city.
    The LLM reads this docstring to decide when to call the tool.
    The @tool decorator converts this into a LangChain Tool object
    with a name, description, and JSON schema derived from the signature.
Schema:      {'description': 'Get the current weather for a city.\nThe LLM reads this docstring to decide when to call the tool.\nThe @tool decorator converts this into a LangChain Tool object\nwith a name, description, and JSON schema derived from the signature.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather_lg', 'type': 'object'}


/var/folders/71/7mh3_2nx4rg1xs9mhzb6zq0r0000gn/T/ipykernel_63486/3963094937.py:18: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"Schema:      {get_weather_lg.args_schema.schema()}")


In [4]:
# ===========================================================================
# MAF tool definition
# ===========================================================================

def get_weather_maf(city: str) -> str:
    """Get the current weather for a city.
    MAF reads the function signature and docstring automatically.
    No decorator needed -- just a plain Python function.
    """
    return f"Sunny in {city}"

# MAF inspects __name__, __doc__, and type annotations at registration time.
# The function is passed directly to the tools= parameter of as_agent().
print(f"Name:        {get_weather_maf.__name__}")
print(f"Docstring:   {get_weather_maf.__doc__.strip().splitlines()[0]}")
print(f"Annotations: {get_weather_maf.__annotations__}")

Name:        get_weather_maf
Docstring:   Get the current weather for a city.
Annotations: {'city': <class 'str'>, 'return': <class 'str'>}


### Key Insight

The underlying function is **identical** in both cases. What differs is the registration mechanism:

- **LangGraph** uses the `@tool` decorator from `langchain_core.tools`. This wraps the function in a `StructuredTool` object that carries a Pydantic schema, integrates with LangChain's tool-calling protocol, and can be passed to `ToolNode` or `bind_tools()`.

- **MAF** uses plain introspection. It reads `__name__`, `__doc__`, and `__annotations__` at agent creation time and generates the tool schema automatically. No wrapper object is created.

**Practical implication:** If you are porting tools between frameworks, the function body stays the same. You only need to add or remove the `@tool` decorator.

## Section 4: State Management

This is where the two frameworks diverge most significantly. State management determines how you persist conversations, debug agent behavior, and handle long-running sessions.

### LangGraph: Explicit State with Reducers

Every LangGraph agent defines its state as a `TypedDict`. The `Annotated[list, add_messages]` pattern tells the framework how to merge updates from different nodes:

```python
class State(TypedDict):
    messages: Annotated[list, add_messages]  # reducer appends correctly
    active_supervisor: str                    # simple overwrite
    pending_code_approval: bool               # simple overwrite
```

Each node returns a **delta** (only the fields it wants to change). The framework merges the delta into the current state using the reducer for that field. This makes state updates composable and predictable.

Persistence is built-in via checkpointers:

```python
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver(sqlite3.connect("checkpoints.db"))
graph = workflow.compile(checkpointer=checkpointer)
```

This gives you automatic persistence after every node execution, full history replay, and the ability to inspect any historical state.

### MAF: Implicit State with Manual Persistence

MAF manages message history internally within the agent object. You do not define a state schema or reducers. For persistence, the workshop implements manual JSON serialization:

```python
from memory_store import load_session, save_session

mem = load_session("router_memory.json", session_id)
conversation_history = mem.get("conversation_history", [])
# ... after each turn ...
save_session("router_memory.json", session_id, {
    "conversation_history": conversation_history,
    "conversation_summary": current_summary,
})
```

This is simpler to start with but requires more manual work for features like history replay or state inspection.

### Comparison Table

| Aspect | LangGraph | MAF |
|--------|-----------|-----|
| State type | Explicit `TypedDict` with typed fields | Implicit message list |
| Reducer | `add_messages` appends and deduplicates | Automatic (internal) |
| Persistence | `SqliteSaver` -- automatic after every node | JSON files -- manual save/load |
| State inspection | Full history replay via checkpointer | Debug logging |
| Custom fields | Add any field to the `TypedDict` | Manage separately in your code |
| Complexity | Higher (but powerful and debuggable) | Lower (but limited visibility) |

## Section 5: Async and Concurrency

The async story is where MAF has a natural advantage.

### MAF: Async-Native

Every MAF agent is async from the ground up. The entry point is always `async def main()` with `asyncio.run(main())`. Tool functions can be either sync or async -- the framework handles both. This makes MAF a natural fit for:

- **Parallel tool execution** via `asyncio.gather()`
- **Web server deployment** -- MAF agents slot directly into FastAPI or similar async frameworks
- **MCP integration** -- the MCP SSE client is async, so MAF connects to it without any adaptation layer

### LangGraph: Sync-First with Async Support

LangGraph's primary API is synchronous (`graph.invoke()`, `graph.stream()`). Async variants exist (`graph.ainvoke()`, `graph.astream()`), but the graph definition itself uses sync node functions by default. When you need async (for example, to call MCP tools), you define async node functions and use the async invocation API.

From the MCP agent in this repository (`mcp_skills_langgraph.py`):

```python
async def tool_node(state: AgentState):
    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = await tool_map[tool_call["name"]].ainvoke(tool_call["args"])
        outputs.append(ToolMessage(content=json.dumps(tool_result), ...))
    return {"messages": outputs}

# Invoked with:
result = await graph.ainvoke({"messages": [("user", query)]})
```

### Impact on MCP Integration

Both frameworks connect to MCP servers using the same async SSE client:

```python
async with sse_client("http://127.0.0.1:8787/sse") as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        # Load tools from the MCP server
```

The difference is in how tools are adapted:

- **LangGraph** uses `langchain_mcp_adapters.tools.load_mcp_tools()` to convert MCP tools into LangChain `Tool` objects.
- **MAF** requires a custom adapter (`make_mcp_tool_func`) that builds Python functions with the correct `__signature__`, `__name__`, and `__doc__` so the framework can generate the tool schema.

See `single-agents/langgraph/mcp_skills_langgraph.py` and `single-agents/maf/mcp_skills_maf.py` for the full implementations.

## Section 6: Human-in-the-Loop

Human-in-the-loop (HITL) is critical for production agents. Both frameworks support it, but with fundamentally different mechanisms.

### LangGraph: Serializable Interrupts

LangGraph uses `interrupt()` from `langgraph.types` to pause graph execution. The interrupt payload is serialized into the checkpoint, and the graph resumes when you call `invoke(Command(resume=...))`. This is the pattern used in the router (`agent-patterns/Langgraph/Router.py`):

```python
from langgraph.types import interrupt, Command

# Inside a graph node -- execution pauses here:
human_response = interrupt({
    "type": "routing_approval",
    "proposed_supervisor": proposed,
    "reason": reason,
    "message": f"Routing to: {proposed.upper()}. Confirm or override.",
})

# Later, from the CLI or a web handler -- execution resumes:
result = router_graph.invoke(
    Command(resume="approved"),
    config={"configurable": {"thread_id": session_id}},
)
```

The router implements **two HITL gates**:
1. **Routing approval** -- confirm or override which supervisor handles the request.
2. **Code approval** -- review generated code before execution.

### MAF: Standard Input

MAF uses Python's built-in `input()` function. The same two HITL gates in the MAF router (`agent-patterns/MAF/Router.py`):

```python
# Routing gate:
human_response = input("Your decision: ").strip().lower()
if not human_response or human_response in ("ok", "yes"):
    return proposed

# Code approval gate:
decision = input("Your decision: ").strip()
phase2 = await resume_code_supervisor(
    user_message=user_message,
    human_input=decision,
    code=phase1["code"],
)
```

### Trade-off Analysis

| Aspect | LangGraph `interrupt()` | MAF `input()` |
|--------|------------------------|---------------|
| Serializable | Yes -- stored in checkpoint | No -- blocks the process |
| Web-deployable | Yes -- resume via API call | Requires refactoring |
| Testable | Yes -- mock the resume value | Requires stdin mocking |
| Simplicity | More boilerplate | Trivial to implement |
| Multiple HITL points | Natural -- each is a node | Manual flow control |

**Key insight:** LangGraph's `interrupt()`/`Command()` pattern makes HITL a first-class, serializable concept. You can pause a graph, shut down the server, restart it days later, and resume from exactly where it stopped. MAF's `input()` works perfectly for CLI tools but requires significant refactoring to deploy behind a web API.

## Section 7: Multi-Agent Router -- The Full Picture

The router pattern is the most complex implementation in this repository, and it best illustrates the architectural differences between the two frameworks.

Both routers do the same thing:
1. Accept a user message.
2. Route it to one of four supervisors (knowledge, code, engineering, chat).
3. Let the human confirm or override the routing decision (HITL 1).
4. Dispatch to the selected supervisor.
5. If it is a code request, show the generated code for review (HITL 2).
6. Return the final answer.

### LangGraph Router Architecture

The LangGraph router is a compiled `StateGraph` with four nodes:

```
START --> router --> routing_gate --> dispatch --+--> END
                                                |
                                                +--> approval_gate --> dispatch
```

- **`RouterState`** carries 9 typed fields including `messages`, `active_supervisor`, `pending_code_approval`, and `conversation_summary`.
- **`SqliteSaver`** persists every state transition to a SQLite database.
- **Routing** uses `llm.with_structured_output(RoutingDecision)` with a Pydantic model.
- **HITL** uses two `interrupt()` calls -- one in `routing_gate_node`, one in `approval_gate_node`.

### MAF Router Architecture

The MAF router is a sequence of async function calls orchestrated in a `while True` loop:

```
while True:
    routing  = await route(user_input, history, summary)
    confirmed = routing_gate(routing)           # HITL 1: input()
    result   = await dispatch(confirmed, ...)   # HITL 2: input() inside dispatch
    history.append(...)
    summary  = await maybe_summarize(history, summary)
    save_session(...)
```

- **State** is a plain Python list (`conversation_history`) and a string (`current_summary`).
- **Persistence** uses `memory_store.py` for JSON file serialization.
- **Routing** uses `OpenAIChatClient().as_agent()` with a prompt that asks for JSON output.
- **Summarization** is an explicit async function that runs when history exceeds a threshold.

### Structural Comparison

| Aspect | LangGraph Router | MAF Router |
|--------|-----------------|------------|
| Architecture | Compiled `StateGraph` with 4 nodes | Sequential async functions |
| State fields | 9 typed fields in `RouterState` | Dict-based, loosely typed |
| Routing model | `with_structured_output(Pydantic)` | JSON parsing from agent response |
| Persistence | `SqliteSaver` (automatic) | `save_session()` to JSON (manual) |
| HITL | `interrupt()` + `Command(resume=)` | `input()` |
| Summarization | Inline in `router_node` | Separate `maybe_summarize()` async function |
| Lines of code | ~490 (Router.py) | ~360 (Router.py) |

> **Source files:** `agent-patterns/Langgraph/Router.py` and `agent-patterns/MAF/Router.py`

## Section 8: Decision Framework

### Choose LangGraph When

- **You need complex, multi-step workflows with branching.** The graph model makes it natural to express conditional transitions, parallel branches, and cycles. If your workflow has more than two or three decision points, the graph makes the topology explicit and auditable.

- **Persistence and replay are important.** `SqliteSaver` gives you automatic checkpointing after every node. You can inspect any historical state, replay from a checkpoint, and resume interrupted sessions -- all without writing persistence code.

- **You want to deploy HITL in a web UI.** The `interrupt()`/`Command()` pattern is serializable. A web server can receive the interrupt payload, render a UI for human review, and resume the graph via an API call. This is not possible with `input()`.

- **You need observability.** LangSmith integration gives you tracing, cost tracking, and visualization of every graph execution. This is valuable for debugging and monitoring production agents.

- **The team is comfortable with graph-based thinking.** If your team has experience with workflow engines, state machines, or dataflow programming, LangGraph will feel natural.

### Choose MAF When

- **You want rapid prototyping with minimal boilerplate.** Going from zero to a working agent takes fewer lines of code. The `as_agent()` API abstracts away the ReAct loop, tool dispatch, and message management.

- **Your agents are Azure-native.** MAF integrates naturally with Azure Functions, Azure AI services, and the broader Microsoft ecosystem. If your deployment target is Azure, MAF reduces friction.

- **The workflow is relatively linear.** If your agent follows a straightforward request-response pattern without complex branching, the graph overhead is unnecessary.

- **You prefer imperative async Python over graph definitions.** Standard `if/else`, `for` loops, and `await` calls are easier to read and debug for developers who are not familiar with graph abstractions.

- **Time-to-first-agent is the priority.** When you need to demonstrate a working prototype quickly, MAF's lower ceremony gets you there faster.

### Decision Tree

Use this as a quick reference when starting a new agent project:

```
Is your workflow complex with branching logic?
|
+-- Yes --> LangGraph
|
+-- No --> Do you need persistent, web-deployable HITL?
    |
    +-- Yes --> LangGraph
    |
    +-- No --> Do you need automatic state persistence and replay?
        |
        +-- Yes --> LangGraph
        |
        +-- No --> Are you deploying on Azure natively?
            |
            +-- Yes --> MAF
            |
            +-- No --> Either works -- pick based on team familiarity
```

Note that these are not hard rules. Both frameworks are capable of solving most agent problems. The decision is about which set of trade-offs better fits your specific constraints.

## Section 9: Summary and Exercises

### Master Comparison Table

| Feature | LangGraph | MAF |
|---------|-----------|-----|
| Philosophy | Declarative graph | Imperative async |
| Agent creation | `StateGraph` + `compile()` | `OpenAIChatClient().as_agent()` |
| Tool definition | `@tool` decorator | Plain Python functions |
| State management | Explicit `TypedDict` with reducers | Implicit message list |
| Persistence | `SqliteSaver` (automatic) | Manual JSON serialization |
| Human-in-the-loop | `interrupt()` / `Command()` | `input()` |
| Async model | Sync-first, async optional | Async-native |
| MCP integration | `langchain_mcp_adapters` | Custom `make_mcp_tool_func` |
| Routing model | `with_structured_output(Pydantic)` | JSON parsing from agent response |
| Lines of code | More (explicit control) | Less (abstracted internals) |
| Debugging | Graph visualization, LangSmith | Logging, print statements |
| Best for | Complex workflows, production HITL | Rapid prototyping, Azure deployments |

---

### Cross-References

Compare any agent pair across the repository:

| Agent | LangGraph | MAF |
|-------|-----------|-----|
| Basic ReAct | `single-agents/langgraph/basic_react_agent.py` | `single-agents/maf/basic_react_agent.py` |
| SQL Agent | `single-agents/langgraph/sql_react_agent.py` | `single-agents/maf/sql_react_agent.py` |
| CSV Analyzer | `single-agents/langgraph/CSV-File-Analyzer.py` | `single-agents/maf/CSV-File-Analyzer.py` |
| Code Interpreter | `single-agents/langgraph/code-interpreter.py` | `single-agents/maf/code-interpreter.py` |
| Agentic RAG | `single-agents/langgraph/Agentic_Rag.py` | `single-agents/maf/Agentic_Rag.py` |
| Reflection Agent | `single-agents/langgraph/Reflection-Agent.py` | `single-agents/maf/Reflection-Agent.py` |
| MCP Skills | `single-agents/langgraph/mcp_skills_langgraph.py` | `single-agents/maf/mcp_skills_maf.py` |
| Multi-Agent Router | `agent-patterns/Langgraph/Router.py` | `agent-patterns/MAF/Router.py` |

---

### Exercises

**(a) Port the SQL agent from LangGraph to MAF (or vice versa)**

Pick one direction. Start by identifying what changes: tool registration, the agent loop, and state handling. What stays the same? The tool function bodies and the system prompt. Measure the line count difference.

**(b) Design a new agent on paper and argue which framework fits better**

Choose a domain (e.g., a customer support agent with escalation, a CI/CD pipeline agent, or a research assistant with web search). Draw the workflow. Identify whether it needs branching, HITL, or persistent state. Use the decision tree from Section 8 to justify your framework choice.

**(c) List 3 features you would need to add to MAF to match LangGraph's router**

Read both `Router.py` files carefully. What does the LangGraph version get for free that the MAF version implements manually? Consider: state checkpointing, interrupt serialization, graph visualization. For each feature, sketch how you would implement it in MAF.